# 01 - Data Exploration
Explorasi PDF raw, cek hasil loading, preprocessong, dan chunking.

In [1]:
import sys
from dotenv import load_dotenv

sys.path.insert(0, '../src')
load_dotenv('../.env')

True

## 1. Load PDF

In [2]:
from core.services.processing.pdf_loader import load_all_pdfs

RAW_DIR = '../storage/raw'
pages = load_all_pdfs(RAW_DIR)

print(f"Total pages loaded: {len(pages)}")
print(f"\nSample page:")
print(f"Source: {pages[0]['metadata']['source']}")
print(f"Page: {pages[0]['metadata']['page']}")
print(f"Langth: {len(pages[0]['text'])} characters")
print(f"Preview: {pages[0]['text'][:200]}...")

Page 80 skipped - content is too short (0 character)
Page 1 skipped - content is too short (27 character)
Page 2 skipped - content is too short (21 character)
Page 2 skipped - content is too short (0 character)
Page 121 skipped - content is too short (0 character)
Page 1 skipped - content is too short (0 character)
Page 2 skipped - content is too short (0 character)
Page 88 skipped - content is too short (0 character)
Page 1 skipped - content is too short (37 character)
Page 3 skipped - content is too short (0 character)
Page 13 skipped - content is too short (0 character)
Page 1 skipped - content is too short (0 character)
Page 3 skipped - content is too short (0 character)
Page 5 skipped - content is too short (2 character)
Page 6 skipped - content is too short (1 character)
Page 7 skipped - content is too short (2 character)
Page 8 skipped - content is too short (3 character)
Page 9 skipped - content is too short (4 character)
Page 10 skipped - content is too short (2 character)
Pag

Total pages loaded: 2638

Sample page:
Source: anthro-pc-manual-v322.pdf
Page: 1
Langth: 780 characters
Preview:  
 
WHO Anthro 
HO Anthro 
HO Anthro 
HO Anthro  
for 
for 
for 
for Personal 
ersonal 
ersonal 
ersonal Computers
omputers
omputers
omputers 
Manual
Manual
Manual
Manual 
                            ...


## 2. Distribution pre Document

In [3]:
from collections import Counter

source_counts = Counter(p['metadata']['source'] for p in pages)
print("Pages per document:")

for source, count in sorted(source_counts.items(), key=lambda x: -x[1]):
    print(f"{source:<50} {count:>4} pages")

Pages per document:
WHO Pocket Book of Hospital Care for Children.pdf   424 pages
WHO child growth standards.pdf                      330 pages
Panduan Stimulasi Tumbuh Kembang (SDIDTK) — Kemenkes RI.pdf  300 pages
Essential nutrition actions mainstreaming nutrition through the life-course (WHO Guidelines on Stunting).pdf  209 pages
PGS Ibu Hamil dan Ibu Menyusui - Merge-1.pdf        184 pages
Indicators for assessing infant and young child feeding practices definitions and measurement methods.pdf  120 pages
Pedoman Gizi Seimbang — Kemenkes RI 2014.pdf        113 pages
WHO Infant and Young Child Feeding.pdf              105 pages
Stranas_Percepatan_Pencegahan_Anak_Kerdil.pdf        97 pages
WHO Guideline for complementary feeding of infants and young children 6-23 months of age.pdf   91 pages
MTBS (Manajemen Terpadu Balita Sakit) — Kemenkes RI.pdf   85 pages
Buku KIA.pdf                                         79 pages
WHO IMCI (Integrated Management of Childhood Illness).pdf   78 page

## 3. Preprocessing

In [4]:
from core.services.processing.preprocessor import preprocess_pages

cleaned = preprocess_pages(pages)
skipped = len(pages) - len(cleaned)

print(f"Before preprocessing: {len(pages)} pages")
print(f"After preprocessing: {len(cleaned)} pages")
print(f"Pages skipped: {skipped} ({skipped/len(pages) * 100:.1f}%)")

Page 1 from Essential nutrition actions mainstreaming nutrition through the life-course (WHO Guidelines on Stunting).pdf skip after cleaning - content is not meaningful enough
Page 3 from Essential nutrition actions mainstreaming nutrition through the life-course (WHO Guidelines on Stunting).pdf skip after cleaning - content is not meaningful enough
Page 209 from Essential nutrition actions mainstreaming nutrition through the life-course (WHO Guidelines on Stunting).pdf skip after cleaning - content is not meaningful enough
Page 33 from Pedoman Gizi Seimbang — Kemenkes RI 2014.pdf skip after cleaning - content is not meaningful enough
Page 3 from Stranas_Percepatan_Pencegahan_Anak_Kerdil.pdf skip after cleaning - content is not meaningful enough
Page 19 from Stranas_Percepatan_Pencegahan_Anak_Kerdil.pdf skip after cleaning - content is not meaningful enough
Page 25 from Stranas_Percepatan_Pencegahan_Anak_Kerdil.pdf skip after cleaning - content is not meaningful enough
Page 27 from Str

Before preprocessing: 2638 pages
After preprocessing: 2615 pages
Pages skipped: 23 (0.9%)


## 4. Chunking

In [5]:
from core.services.processing.chunker import chunk_pages

chunks = chunk_pages(cleaned, chunk_size=512, chunk_overlap=128)

print(f"Total chunks generated: {len(chunks)}")
print(f"Avg chunks per page: {len(chunks) / len(cleaned):.1f}")

# length distribution of chunk 
lengths = [len(c['text']) for c in chunks]

print(f"\nChunk length stats:")
print(f"Min: {min(lengths)} chars")
print(f"Max: {max(lengths)} chars")
print(f"Avg: {sum(lengths) / len(lengths):.0f} chars")

Total chunks generated: 13812
Avg chunks per page: 5.3

Chunk length stats:
Min: 48 chars
Max: 512 chars
Avg: 450 chars


## 5. Sample Chunks

In [6]:
seen_sources = set()
samples = []

for chunk in chunks:
    src = chunk['metadata']['source']
    if src not in seen_sources:
        samples.append(chunk)
        seen_sources.add(src)
    if len(samples) == 3:
        break

for i, chunk in enumerate(samples):
    print(f"--- Sample {i+1} ---")
    print(f"Source: {chunk['metadata']['source']}")
    print(f"Page: {chunk['metadata']['page']}")
    print(f"Length: {len(chunk['text'])} chars")
    print(f"text: {chunk['text'][:300]}...")
    print()

--- Sample 1 ---
Source: anthro-pc-manual-v322.pdf
Page: 1
Length: 499 chars
text: WHO Anthro 
HO Anthro 
HO Anthro 
HO Anthro 
for 
for 
for 
for Personal 
ersonal 
ersonal 
ersonal Computers
omputers
omputers
omputers 
Manual
Manual
Manual
Manual 
 
 
 
 
 
 
Software 
oftware 
oftware 
oftware for assess
 assess
 assess
 assessing
ing
ing
ing 
growth and development
growth and ...

--- Sample 2 ---
Source: Buku KIA.pdf
Page: 1
Length: 244 chars
text: Nama Ibu
Tanggal dikeluarkannya Buku
Provinsi
Dikeluarkan oleh Fasilitas Kesehatan
Kab./Kota
Bawa buku ini setiap kali mengunjungi Posyandu, fasilitas kesehatan, kelas ibu, 
BKB dan PAUD. Gunakan dari masa kehamilan sampai anak berumur 6 tahun....

--- Sample 3 ---
Source: CNR 2025 - Feeding Profit - Data Tables- English - FINAL.pdf.pdf
Page: 2
Length: 450 chars
text: Acknowledgements
This publication was prepared by Data and Analytics Section of the UNICEF Data, Analytics, 
Planning and Monitoring Division in collaboration with the UNI